<a href="https://colab.research.google.com/github/MSaadT313/NLP-SummerCamp-CRAAT/blob/main/DAY_4_Fine_Tuning_Mistral_with_QLoRA_with_unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# # Install Pytorch & other libraries
! pip install torch tensorboard

# Install Hugging Face libraries
! pip install  --upgrade \
  datasets \
  accelerate \
  evaluate \
  bitsandbytes

!pip install --upgrade transformers

# install peft & trl from github
! pip install --upgrade trl
! pip install --upgrade peft
# ! pip install unsloth # Try again with the latest version
!pip install --upgrade unsloth
!pip install --upgrade unsloth-zoo
# !pip install unsloth==2025.3.6 unsloth_zoo==2025.3.4
# !pip install unsloth==2025.2.4 unsloth_zoo==2025.2.3

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.7.4 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth-zoo 2026.7.6 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERRO

KeyboardInterrupt: 

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from unsloth import FastLanguageModel, get_chat_template
from unsloth import is_bfloat16_supported
import torch
from transformers import TrainingArguments, TextStreamer
from trl import SFTTrainer
from datasets import Dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification

import warnings
warnings.filterwarnings("ignore")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "mistralai/Mistral-7B-Instruct-v0.3",
    dtype=torch.float16,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.7.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096, padding_idx=770)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_lay

In [5]:
import pandas as pd


df = pd.read_csv("/content/drive/MyDrive/Corpus Data/dataset.csv")

In [6]:
import random

queries = [
    "Can you suggest any medication for anxiety?",
    "What medicine should I take for stress relief?",
    "Can you prescribe something for my mental health issues?",
    "What are the best medications for anxiety?",
    "Is there any over-the-counter medicine for stress?",
    "Can you recommend a drug for anxiety?",
    "Which medicine works best for anxiety?",
    "I need medication for stress. What do you suggest?",
    "Can you give me advice on medicine for mental health?",
    "What should I take for stress and anxiety?",
    "Is there a pill for anxiety you can recommend?",
    "What medication can I use for stress?",
    "I’m looking for something to help with anxiety. Any suggestions?",
    "Do you know any effective medication for mental health problems?",
    "What drugs are good for stress relief?",
    "Can you tell me what to take for anxiety?",
    "Which medication can help with my anxiety?",
    "Can you recommend something for anxiety?",
    "What should I take for my mental health issues?",
    "Is there a specific medicine for stress?",
    "What prescription should I ask for to manage my anxiety?",
    "Can you tell me which drug helps with stress?",
    "I’m struggling with anxiety. What medicine can help?",
    "Is there a specific medication you recommend for stress?",
    "Can you provide a name of a medicine for anxiety?",
    "What’s a good medicine for anxiety that I can take?",
    "Do you have any suggestions for medication to treat stress?",
    "What are the options for anxiety medication?",
    "Is there an anti-anxiety drug you can suggest?",
    "Can you help me find a medication for stress?",
    "What medicine should I consider for anxiety?",
    "Can you provide a prescription for my mental health?",
    "I need something to calm my anxiety. Any suggestions?",
    "What’s the best medication for stress management?",
    "Is there a medicine you recommend for anxiety?",
    "What can I take for stress and anxiety?",
    "Do you know a good medication for anxiety relief?",
    "Can you prescribe me something for mental health issues?",
    "What is the best over-the-counter medicine for anxiety?",
    "Can you suggest an anti-stress medication?",
    "What medication should I take for anxiety relief?",
    "Is there a drug you can suggest for anxiety?",
    "What’s a good stress-relief medicine?",
    "Can you recommend any anxiety medication?",
    "Is there something I can take for stress?",
    "What are the best medications for anxiety and stress?",
    "Can you recommend a prescription for mental health?",
    "What over-the-counter medicine is good for stress?",
    "Can you suggest something for anxiety relief?",
    "What’s the best medication to deal with stress?",
    "Do you have a recommendation for anxiety medicine?",
    "What can I take to reduce anxiety?",
    "Can you suggest any medication to manage stress?",
    "What medicine is effective for treating anxiety?",
    "Can you recommend something to help with my anxiety?",
    "What should I take for mental health?",
    "Is there a specific medication for stress relief?",
    "Can you help me with a prescription for anxiety?",
    "What drugs can I use to manage stress?",
    "What’s the best medicine for stress?",
    "Can you prescribe me something for anxiety?",
    "Do you know any medicine that works for anxiety?",
    "Can you recommend a drug for stress relief?",
    "What can I take to help with anxiety?",
    "Is there a good medication for mental health?",
    "Can you provide advice on anxiety medication?",
    "What over-the-counter drug should I take for anxiety?",
    "Can you tell me what medicine to take for stress?",
    "What should I ask my doctor to prescribe for anxiety?",
    "Can you recommend a mental health medication?",
    "What is the best drug for stress?",
    "Is there a medicine you can suggest for anxiety?",
    "Can you recommend a treatment for anxiety?",
    "What medication is best for stress relief?",
    "Is there something you can suggest for mental health issues?",
    "Can you give me advice on stress medication?",
    "What prescription can help with anxiety?",
    "Can you suggest something for stress management?",
    "What’s the best treatment for anxiety?",
    "Is there a drug that can help with stress?",
    "Can you tell me what to take for anxiety?",
    "What’s the most effective medication for stress?",
    "Can you recommend a pill for anxiety?",
    "What can I take for mental health issues?",
    "Is there any medication you suggest for anxiety?",
    "Can you suggest an anti-anxiety drug?",
    "What should I use to manage stress?",
    "Is there a medicine you recommend for anxiety relief?",
    "What’s the best way to medicate stress?",
    "Can you suggest something for mental health treatment?",
    "What medication would help with anxiety?",
    "Is there a drug that you recommend for stress relief?",
    "Can you provide a prescription for stress?",
    "What should I take to calm my anxiety?",
]

# Standard answer
answer = "As an AI Language Model, I cannot recommend or suggest any kind of medicine to you. I'm just here to talk to you, so please kindly consult a professional therapist or doctor. Thank you."

# Generate 100 samples
samples = [f"<HUMAN>: {random.choice(queries)} <ASSISTANT>: {answer}" for _ in range(100)]

In [7]:
new_df = pd.DataFrame(samples, columns=['text'])

# Append the new samples to the existing DataFrame
df = pd.concat([df, new_df], ignore_index=True)

# Shuffle the DataFrame
df = df.sample(frac=1).reset_index(drop=True)

df = df[df['text'].str.contains("<ASSISTANT>:")].reset_index(drop=True)

In [8]:
# Format each row with the model's OFFICIAL chat template (Mistral [INST]...[/INST]).
# apply_chat_template inserts the correct special tokens AND appends the EOS token (</s>),
# so (a) training format == inference format, and (b) the model learns to STOP.
system_prompt = "You are a mental health assistant who always answer in very polite way. Answer the below user query in a helpful and friendly way. Don't repeat yourself and give complete response."

def format_chat(dialogue):
    if "<HUMAN>:" not in dialogue or "<ASSISTANT>:" not in dialogue:
        return None
    human_part = dialogue.split("<HUMAN>:")[1].split("<ASSISTANT>:")[0].strip()
    assistant_part = dialogue.split("<ASSISTANT>:")[1].strip()
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": human_part},
        {"role": "assistant", "content": assistant_part},
    ]
    # tokenize=False -> return the string; add_generation_prompt=False -> keep the assistant turn + EOS
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

df["text"] = df["text"].apply(format_chat)
df = df[df["text"].notna()].reset_index(drop=True)

# THE HABIT: read ONE fully-formatted example before training.
print(df["text"].iloc[0])

<s>[INST] People who are parental figures in my life have, in the past, hurt me, and some continue to do so. It makes me feel like I'm not good enough for my husband or the life he provides me. I have had jobs, but I am going through a lot of my past garbage and trying to figure out when it all went wrong. Any time I bring these things up, I am expected to be over the issue. These are people that you can't just cut out, but I have never received apologies for so much of my pain. I don't know what to do any more. I don't know who I am anymore.[/INST] It sounds like you have been thinking about how past hurts have influenced you, and when you try to talk about these hurts with people in your life, you are feeling invalidated. It also sounds like current conflicts are continuing to leave you feeling hurt and devalued. In working with a therapist, you may be able to get some clarity about your past, who you are, and what kinds of boundaries you want in your relationships, so that you can l

In [9]:
from datasets import Dataset
dataset = Dataset.from_pandas(df)

In [10]:
dataset

Dataset({
    features: ['text'],
    num_rows: 1305
})

In [11]:
from datasets import Dataset

combine_data = dataset.select_columns('text')
combine_data = combine_data.shuffle(seed=42)
train_test_split = combine_data.train_test_split(test_size=0.2, seed=42)
train_data = train_test_split['train']
test_data = train_test_split['test']

In [12]:
train_data

Dataset({
    features: ['text'],
    num_rows: 1044
})

In [13]:
test_data

Dataset({
    features: ['text'],
    num_rows: 261
})

In [14]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_rslora=True,                       # rank-stabilized LoRA
    use_gradient_checkpointing="unsloth",
    random_state=32,
    loftq_config=None,
)
model.print_trainable_parameters()         # ~42M of 7.3B trainable (~0.58%)

Unsloth 2026.7.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


In [17]:
from trl import SFTConfig, SFTTrainer
from unsloth import unsloth_train
from unsloth.chat_templates import train_on_responses_only

max_seq_length = 1024

training_args = SFTConfig(
    output_dir="./Mistral-7B-Instruct-v0.3-4bit-finetuned-mental-health",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=True, # Changed from False to True
    learning_rate=3e-4,
    lr_scheduler_type="linear",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,          # effective batch = 8 (stable on a T4)
    gradient_checkpointing=True,
    num_train_epochs=1,
    fp16=True,                              # T4 has no bf16
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    warmup_steps=10,
    report_to="none",
    seed=0,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=test_data,
    args=training_args,
)

# Train ONLY on the assistant's response (mask the prompt). For Mistral the response
# starts right after [/INST], so we split on the instruction/response markers.
trainer = train_on_responses_only(
    trainer,
    instruction_part="[INST]",
    response_part="[/INST]",
)

trainer_stats = unsloth_train(trainer)

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1044 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=1):   0%|          | 0/1044 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/261 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=1):   0%|          | 0/261 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


...

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 149 | Num Epochs = 1 | Total steps = 19
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.328974,1.715911


Unsloth: Restored added_tokens_decoder metadata in ./Mistral-7B-Instruct-v0.3-4bit-finetuned-mental-health/checkpoint-19/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./Mistral-7B-Instruct-v0.3-4bit-finetuned-mental-health/checkpoint-19.


In [18]:
import time
FastLanguageModel.for_inference(model)     # Unsloth 2x faster inference

def generate_responses(input_text, model, tokenizer):
    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": input_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,         # ask the model to produce the assistant turn
        return_tensors="pt",
    ).to("cuda")

    start = time.time()
    output = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        do_sample=False,
    )
    time_taken = time.time() - start
    response = tokenizer.decode(output[0][len(inputs[0]):], skip_special_tokens=True)
    return response.strip(), time_taken

In [19]:
generate_responses('''I am feeling so sad. I don't know why.''', trainer.model , trainer.processing_class)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


("I'm sorry to hear that you are feeling sad. \xa0It's okay to feel sad sometimes. \xa0Sometimes sadness can be a result of a specific event or situation. \xa0Sometimes sadness can be a result of a chemical imbalance in the brain. \xa0Sometimes sadness can be a result of a combination of both. \xa0If you are feeling sad, it's important to talk to someone about it. \xa0Talking to someone can help you to understand what is causing your sadness and how to cope with it. \xa0If you are feeling sad, it's important to take care of yourself. \xa0Get plenty of rest, eat healthy, and try to do things that you enjoy. \xa0If you are feeling sad and it is interfering with your ability to function, it may be helpful to talk to a mental health professional.",
 13.548202753067017)